# Bluesky Hardware Smoke

Interactive runbook for validating the `GeecsBluesky` backend without the GUI.

This notebook uses `BlueskyScanner`, so running a scan cell can create a normal GEECS `scans/Scan###` folder through the scanner-side scan-number claim path. It can also move hardware, command shot control, and fire shots when those sections are enabled.

The default parameter block is set up for a real strict-shot-control smoke test: `ACQUISITION_MODE = "strict_shot_control"`, shot control enabled, and an `ARMED` state defined. The scan will still refuse to run until the operator sets the confirmation string after reviewing the exact shot-control writes.


## Operator Checklist

- Confirm lab network access and GEECS DB access.
- Confirm the detector device is on and publishing TCP updates.
- Review the shot-control preflight output before running any scan cell.
- Set `SHOT_CONTROL_CONFIRMATION = "CONFIRM_SHOT_CONTROL"` only after the listed DG645 writes match the live hardware state you intend to test.
- Leave `ENABLE_MOTOR_SCAN = False` unless it is safe to move the configured motor range.
- Leave `ENABLE_NONSCALE_SAVE = False` unless you want native device files written under the claimed scan folder.
- Use the cleanup cell if you interrupt a scan manually.


In [ ]:
# Operator-editable parameters

EXPERIMENT = "Undulator"

# Acquisition modes: "strict_shot_control" or "free_run_time_sync".
# Strict with shot control enabled exercises the plan-owned single-shot path
# only when SHOT_CONTROL_INFO defines a non-empty ARMED state.
ACQUISITION_MODE = "strict_shot_control"

# Primary detector used by the default NOSCAN smoke test.
DETECTOR_DEVICE = "UC_TopView"
DETECTOR_VARIABLES = ["acq_timestamp"]
DETECTOR_SYNCHRONOUS = True
ENABLE_NONSCALE_SAVE = False

# NOSCAN statistics collection. This still claims a Scan### folder when run.
ENABLE_NOSCAN = True
NOSCAN_SHOTS = 3

# Optional STANDARD scan. Keep disabled until the motor range is safe.
ENABLE_MOTOR_SCAN = False
MOTOR_DEVICE = "U_ESP_JetXYZ"
MOTOR_VARIABLE = "Position.Axis 1"
MOTOR_START = 4.0
MOTOR_END = 5.0
MOTOR_STEP = 0.5
STANDARD_SHOTS_PER_STEP = 3

# Real strict-shot-control testing is enabled here, but guarded by an explicit
# confirmation string. Leave the confirmation blank until the preflight output
# matches the current hardware state you intend to command.
ENABLE_SHOT_CONTROL = True
SHOT_CONTROL_REQUIRED_CONFIRMATION = "CONFIRM_SHOT_CONTROL"
SHOT_CONTROL_CONFIRMATION = ""
SHOT_CONTROL_INFO = {
    "device": "U_DG645_ShotControl",
    "variables": {
        "Trigger.ExecuteSingleShot": {
            "OFF": "",
            "SCAN": "",
            "ARMED": "",
            "SINGLESHOT": "on",
            "STANDBY": "",
        },
        "Trigger.Source": {
            "OFF": "Single shot external rising edges",
            "SCAN": "External rising edges",
            "ARMED": "Single shot external rising edges",
            "STANDBY": "External rising edges",
        },
        # Add timing-output amplitude/state variables here if the deployed
        # timing YAML needs additional writes for a true full-power ARMED state.
    },
}
SHOT_CONTROL_REVIEW_STATES = ("OFF", "STANDBY", "ARMED", "SINGLESHOT", "SCAN")

# The scanner derives shots_per_step as round(rep_rate_hz * wait_time).
REP_RATE_HZ = 1.0

# Polling interval for notebook progress output.
POLL_INTERVAL_S = 0.5

In [ ]:
import logging
import os
import time
from types import SimpleNamespace

from geecs_bluesky.db.geecs_db import GeecsDb
from geecs_bluesky.scanner_bridge import BlueskyScanner

logging.basicConfig(level=logging.INFO, format="%(name)s %(levelname)s %(message)s")
logger = logging.getLogger("bluesky_hardware_smoke")

last_scanner = None

os.environ["GEECS_BLUESKY_ACQUISITION_MODE"] = ACQUISITION_MODE
print(f"Using acquisition mode: {ACQUISITION_MODE}")

In [ ]:
def make_devices_config() -> dict:
    """Build the BlueskyScanner save-device config from operator parameters."""
    return {
        DETECTOR_DEVICE: {
            "variable_list": list(DETECTOR_VARIABLES),
            "synchronous": DETECTOR_SYNCHRONOUS,
            "save_nonscalar_data": ENABLE_NONSCALE_SAVE,
        }
    }


def shot_control_writes_for_state(state: str) -> dict[str, str]:
    """Return non-empty shot-control writes for a named state."""
    return {
        variable: values[state]
        for variable, values in SHOT_CONTROL_INFO.get("variables", {}).items()
        if values.get(state)
    }


def print_shot_control_preflight() -> None:
    """Print the exact shot-control writes this notebook can command."""
    if not ENABLE_SHOT_CONTROL:
        print("Shot control: disabled")
        return

    print("Shot control: enabled")
    print(f"  device: {SHOT_CONTROL_INFO.get('device')}")
    print("  writes by state:")
    for state in SHOT_CONTROL_REVIEW_STATES:
        writes = shot_control_writes_for_state(state)
        if not writes:
            print(f"    {state}: <no-op>")
            continue
        print(f"    {state}:")
        for variable, value in writes.items():
            print(f"      {variable} = {value!r}")


def require_shot_control_confirmation() -> None:
    """Refuse to command shot control until the operator confirms values."""
    if not ENABLE_SHOT_CONTROL:
        return

    print_shot_control_preflight()
    if ACQUISITION_MODE == "strict_shot_control" and not shot_control_writes_for_state(
        "ARMED"
    ):
        raise RuntimeError(
            "strict_shot_control with shot control enabled requires a non-empty "
            "ARMED state to exercise plan-owned single-shot acquisition"
        )
    if SHOT_CONTROL_CONFIRMATION != SHOT_CONTROL_REQUIRED_CONFIRMATION:
        raise RuntimeError(
            "Shot control is enabled. Review the writes above, then set "
            f"SHOT_CONTROL_CONFIRMATION = {SHOT_CONTROL_REQUIRED_CONFIRMATION!r} "
            "in the parameter cell before running acquisition."
        )


def make_exec_config(
    scan_mode: str,
    *,
    shots_per_step: int,
    device_var: str | None = None,
    start: float = 0.0,
    end: float = 0.0,
    step: float = 1.0,
    description: str = "",
) -> SimpleNamespace:
    """Build a duck-typed ScanExecutionConfig accepted by BlueskyScanner."""
    wait_time = max(1.0 / REP_RATE_HZ, shots_per_step / REP_RATE_HZ)
    scan_config = SimpleNamespace(
        scan_mode=scan_mode,
        device_var=device_var,
        start=start,
        end=end,
        step=step,
        wait_time=wait_time,
        additional_description=description,
    )
    options = SimpleNamespace(
        rep_rate_hz=REP_RATE_HZ, acquisition_mode=ACQUISITION_MODE
    )
    save_config = SimpleNamespace(Devices=make_devices_config())
    return SimpleNamespace(
        scan_config=scan_config, options=options, save_config=save_config
    )


def wait_for_scan(scanner: BlueskyScanner) -> None:
    """Poll scanner progress until the background scan thread exits."""
    while scanner.is_scanning_active():
        print(f"progress={scanner.estimate_current_completion() * 100:5.1f}%", end="\r")
        time.sleep(POLL_INTERVAL_S)
    print("progress=100.0%")


def run_scanner(
    exec_config: SimpleNamespace, *, label: str
) -> tuple[BlueskyScanner, list[dict], list[dict]]:
    """Run one scan and collect start/event documents for inspection."""
    events: list[dict] = []
    starts: list[dict] = []
    require_shot_control_confirmation()
    scanner = BlueskyScanner(
        experiment_dir=EXPERIMENT,
        shot_control_information=SHOT_CONTROL_INFO if ENABLE_SHOT_CONTROL else None,
    )

    def collect(name: str, doc: dict) -> None:
        if name == "start":
            starts.append(doc)
        elif name == "event":
            events.append(doc)
            print(f"event {len(events):03d}: keys={sorted(doc.get('data', {}))}")

    scanner._RE.subscribe(
        collect
    )  # Diagnostic hook until a public document stream exists.
    print(f"\n=== {label} ===")
    ok = scanner.reinitialize(exec_config=exec_config)
    if not ok:
        raise RuntimeError(f"{label}: scanner reinitialize returned False")
    scanner.start_scan_thread()
    wait_for_scan(scanner)
    return scanner, starts, events


def summarize_run(
    starts: list[dict], events: list[dict], *, expected_events: int | None = None
) -> None:
    """Print and assert high-level run metadata and event-count checks."""
    start = starts[-1] if starts else {}
    print("\nRun summary")
    print(f"  scan_id: {start.get('scan_id')}")
    print(f"  scan_number: {start.get('scan_number')}")
    print(f"  scan_folder: {start.get('scan_folder')}")
    print(f"  events: {len(events)}")
    if expected_events is not None:
        print(f"  expected_events: {expected_events}")
        assert len(events) == expected_events, (
            f"expected {expected_events}, got {len(events)}"
        )
    assert starts, "no Bluesky start document captured"
    assert events, "no event documents captured"
    assert start.get("scan_folder"), (
        "scan_folder missing; scan number claim may have failed"
    )
    assert any("acq_timestamp" in key for key in events[-1].get("data", {})), (
        "no acq_timestamp-like event key found"
    )
    print("  checks: passed")

In [ ]:
# Connectivity preflight. This does not create a scan folder.
for device in [DETECTOR_DEVICE] + ([MOTOR_DEVICE] if ENABLE_MOTOR_SCAN else []):
    host, port = GeecsDb.find_device(device)
    print(f"{device}: {host}:{port}")

## NOSCAN Smoke

This is the safest first live test. It still claims a GEECS scan number and creates a `scans/Scan###` folder when the scanner starts.

In [ ]:
if ENABLE_NOSCAN:
    noscan_config = make_exec_config(
        "noscan",
        shots_per_step=NOSCAN_SHOTS,
        description="Notebook Bluesky NOSCAN hardware smoke",
    )
    last_scanner, noscan_starts, noscan_events = run_scanner(
        noscan_config, label="NOSCAN"
    )
    summarize_run(noscan_starts, noscan_events, expected_events=NOSCAN_SHOTS)
else:
    print("NOSCAN disabled")

## Optional STANDARD Motor Scan

This section moves hardware. It runs only when `ENABLE_MOTOR_SCAN = True` in the parameter cell.

In [ ]:
if ENABLE_MOTOR_SCAN:
    standard_config = make_exec_config(
        "standard",
        shots_per_step=STANDARD_SHOTS_PER_STEP,
        device_var=f"{MOTOR_DEVICE}:{MOTOR_VARIABLE}",
        start=MOTOR_START,
        end=MOTOR_END,
        step=MOTOR_STEP,
        description="Notebook Bluesky STANDARD hardware smoke",
    )
    positions = (
        int(round(abs(MOTOR_END - MOTOR_START) / abs(MOTOR_STEP))) + 1
        if MOTOR_STEP
        else 1
    )
    expected = max(1, positions) * STANDARD_SHOTS_PER_STEP
    last_scanner, standard_starts, standard_events = run_scanner(
        standard_config, label="STANDARD"
    )
    summarize_run(standard_starts, standard_events, expected_events=expected)
else:
    print("STANDARD motor scan disabled")

## Cleanup / Abort

Run this cell if you interrupted a scan or want to force cleanup of the last scanner instance.

In [ ]:
if last_scanner is not None and last_scanner.is_scanning_active():
    last_scanner.stop_scanning_thread()
    print("Stop requested for active scanner")
else:
    print("No active scanner to stop")